<a href="https://colab.research.google.com/github/NazaninParvizi/Assignment-4--DLG-iDLG-Attacksagainst-baseline-defenses/blob/main/dlg_attack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Change to 'TkAgg' or remove for interactive plots
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
import warnings
import time
import json

In [ ]:
import os
from google.colab import files

file_name = "mnist_train.csv"

print(f"Please upload the '{file_name}' file.")
uploaded = files.upload()

if file_name in uploaded:
  print(f"'{file_name}' uploaded successfully.")
else:
  print(f"'{file_name}' was not uploaded. Please ensure you select the correct file.")

Please upload the 'mnist_train.csv' file.


Saving mnist_train.csv to mnist_train.csv
'mnist_train.csv' uploaded successfully.


In [ ]:

"""
DLG (Deep Leakage from Gradients) Attack Implementation
with Multiple Defense Mechanisms on MNIST Dataset

This implementation includes:
1. DLG Attack
2. Defense Baselines:
   - Differential Privacy (DP)
   - Gradient Clipping
   - Gradient Sparsification (Top-K)
   - Additive Noise
   - No Defense (baseline)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List, Optional
import copy


# ================== Model Definition ==================
class SimpleNet(nn.Module):
    """Simple CNN for MNIST classification"""
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)


# ================== Defense Mechanisms ==================
class DefenseMechanism:
    """Base class for defense mechanisms"""
    def __init__(self, name: str):
        self.name = name

    def apply(self, gradients: List[torch.Tensor]) -> List[torch.Tensor]:
        """Apply defense to gradients"""
        raise NotImplementedError


class NoDefense(DefenseMechanism):
    """No defense - baseline"""
    def __init__(self):
        super().__init__("No Defense")

    def apply(self, gradients: List[torch.Tensor]) -> List[torch.Tensor]:
        return gradients


class DifferentialPrivacy(DefenseMechanism):
    """Differential Privacy with Gaussian noise"""
    def __init__(self, noise_multiplier: float = 1.0, clip_norm: float = 1.0):
        super().__init__(f"DP (σ={noise_multiplier}, C={clip_norm})")
        self.noise_multiplier = noise_multiplier
        self.clip_norm = clip_norm

    def apply(self, gradients: List[torch.Tensor]) -> List[torch.Tensor]:
        defended_grads = []
        for grad in gradients:
            # Clip gradient
            grad_norm = torch.norm(grad)
            if grad_norm > self.clip_norm:
                grad = grad * (self.clip_norm / grad_norm)

            # Add Gaussian noise
            noise = torch.randn_like(grad) * self.noise_multiplier * self.clip_norm
            defended_grads.append(grad + noise)

        return defended_grads


class GradientClipping(DefenseMechanism):
    """Gradient Clipping"""
    def __init__(self, clip_norm: float = 1.0):
        super().__init__(f"Clipping (C={clip_norm})")
        self.clip_norm = clip_norm

    def apply(self, gradients: List[torch.Tensor]) -> List[torch.Tensor]:
        defended_grads = []
        for grad in gradients:
            grad_norm = torch.norm(grad)
            if grad_norm > self.clip_norm:
                grad = grad * (self.clip_norm / grad_norm)
            defended_grads.append(grad)

        return defended_grads


class GradientSparsification(DefenseMechanism):
    """Gradient Sparsification (Top-K)"""
    def __init__(self, sparsity: float = 0.1):
        super().__init__(f"Sparsification (k={sparsity})")
        self.sparsity = sparsity

    def apply(self, gradients: List[torch.Tensor]) -> List[torch.Tensor]:
        defended_grads = []
        for grad in gradients:
            # Flatten gradient
            flat_grad = grad.flatten()
            k = max(1, int(flat_grad.numel() * self.sparsity))

            # Get top-k values
            _, top_indices = torch.topk(torch.abs(flat_grad), k)

            # Create sparse gradient
            sparse_grad = torch.zeros_like(flat_grad)
            sparse_grad[top_indices] = flat_grad[top_indices]

            defended_grads.append(sparse_grad.reshape(grad.shape))

        return defended_grads


class AdditiveNoise(DefenseMechanism):
    """Simple Additive Gaussian Noise"""
    def __init__(self, noise_std: float = 0.1):
        super().__init__(f"Additive Noise (σ={noise_std})")
        self.noise_std = noise_std

    def apply(self, gradients: List[torch.Tensor]) -> List[torch.Tensor]:
        defended_grads = []
        for grad in gradients:
            noise = torch.randn_like(grad) * self.noise_std
            defended_grads.append(grad + noise)

        return defended_grads


# ================== DLG Attack ==================
class DLGAttack:
    """Deep Leakage from Gradients Attack"""
    def __init__(self, model: nn.Module, device: str = 'cpu'):
        self.model = model
        self.device = device

    def attack(
        self,
        original_gradients: List[torch.Tensor],
        original_label: torch.Tensor,
        num_iterations: int = 300,
        lr: float = 0.1,
        defense: Optional[DefenseMechanism] = None
    ) -> Tuple[torch.Tensor, List[float]]:
        """
        Perform DLG attack to reconstruct input from gradients

        Args:
            original_gradients: True gradients from the model
            original_label: True label
            num_iterations: Number of optimization iterations
            lr: Learning rate for reconstruction
            defense: Defense mechanism to apply

        Returns:
            Reconstructed image and loss history
        """
        # Apply defense if provided
        if defense is not None:
            defended_gradients = defense.apply(original_gradients)
        else:
            defended_gradients = original_gradients

        # Initialize dummy data and label
        dummy_data = torch.randn((1, 1, 28, 28), requires_grad=True, device=self.device)
        dummy_label = original_label.clone().detach().to(self.device)

        optimizer = torch.optim.LBFGS([dummy_data], lr=lr)

        loss_history = []

        for iteration in range(num_iterations):
            def closure():
                optimizer.zero_grad()

                # Forward pass with dummy data
                dummy_pred = self.model(dummy_data)
                dummy_loss = F.nll_loss(dummy_pred, dummy_label)

                # Compute gradients
                dummy_gradients = torch.autograd.grad(
                    dummy_loss, self.model.parameters(), create_graph=True
                )

                # Compute gradient matching loss
                grad_diff = 0
                for gx, gy in zip(dummy_gradients, defended_gradients):
                    grad_diff += ((gx - gy) ** 2).sum()

                grad_diff.backward()

                return grad_diff

            loss = optimizer.step(closure)
            loss_history.append(loss.item())

            if iteration % 50 == 0:
                print(f"  Iteration {iteration}, Loss: {loss.item():.4f}")

        return dummy_data.detach(), loss_history


# ================== Evaluation Metrics ==================
def calculate_psnr(original: torch.Tensor, reconstructed: torch.Tensor) -> float:
    """Calculate Peak Signal-to-Noise Ratio"""
    mse = F.mse_loss(original, reconstructed)
    if mse == 0:
        return float('inf')
    max_pixel = 1.0
    psnr = 20 * torch.log10(max_pixel / torch.sqrt(mse))
    return psnr.item()


def calculate_mse(original: torch.Tensor, reconstructed: torch.Tensor) -> float:
    """Calculate Mean Squared Error"""
    return F.mse_loss(original, reconstructed).item()


# ================== Plotting Functions ==================
def plot_comparison(
    original_image: torch.Tensor,
    results: dict,
    save_path: str = 'dlg_comparison.png'
):
    """Plots original and reconstructed images for comparison"""
    num_defenses = len(results)
    fig, axes = plt.subplots(1, num_defenses + 1, figsize=(3 * (num_defenses + 1), 3))

    # Plot original image
    axes[0].imshow(original_image.squeeze().cpu().numpy(), cmap='gray')
    axes[0].set_title("Original Image")
    axes[0].axis('off')

    # Plot reconstructed images
    for i, (defense_name, (reconstructed_image, psnr, mse)) in enumerate(results.items()):
        ax = axes[i + 1]
        ax.imshow(reconstructed_image.squeeze().cpu().numpy(), cmap='gray')
        ax.set_title(f"{defense_name}\nPSNR: {psnr:.2f}dB\nMSE: {mse:.4f}")
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()


def plot_loss_curves(
    loss_histories: dict,
    save_path: str = 'dlg_loss_curves.png'
):
    """Plots the loss history for each defense mechanism"""
    plt.figure(figsize=(10, 6))
    for defense_name, history in loss_histories.items():
        plt.plot(history, label=defense_name)
    plt.title("DLG Attack Loss Curves")
    plt.xlabel("Iteration")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path)
    plt.close()


# ================== Main Experiment ==================
def run_experiment(
    sample_idx: int = 0,
    num_iterations: int = 300,
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
):
    """Run complete DLG attack experiment with multiple defenses"""

    print(f"Using device: {device}")
    print("=" * 70)

    # Load MNIST dataset
    transform = transforms.Compose([
        transforms.ToTensor(),
    ])

    dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)

    # Get a sample
    original_image, original_label = dataset[sample_idx]
    original_image = original_image.unsqueeze(0).to(device)
    original_label = torch.tensor([original_label]).to(device)

    print(f"Original Label: {original_label.item()}")
    print("=" * 70)

    # Initialize model
    model = SimpleNet().to(device)
    model.eval()

    # Compute original gradients
    criterion = nn.NLLLoss()
    output = model(original_image)
    loss = criterion(output, original_label)
    original_gradients = torch.autograd.grad(loss, model.parameters())
    original_gradients = [g.detach().clone() for g in original_gradients]

    # Define defense mechanisms
    defenses = [
        NoDefense(),
        DifferentialPrivacy(noise_multiplier=0.5, clip_norm=1.0),
        DifferentialPrivacy(noise_multiplier=1.0, clip_norm=1.0),
        GradientClipping(clip_norm=1.0),
        GradientSparsification(sparsity=0.1),
        GradientSparsification(sparsity=0.01),
        AdditiveNoise(noise_std=0.1),
    ]

    # Run attack with each defense
    attacker = DLGAttack(model, device)
    results = {}
    loss_histories = {}

    for defense in defenses:
        print(f"\nTesting: {defense.name}")
        print("-" * 70)

        reconstructed, loss_history = attacker.attack(
            original_gradients,
            original_label,
            num_iterations=num_iterations,
            defense=defense
        )

        # Calculate metrics
        psnr = calculate_psnr(original_image, reconstructed)
        mse = calculate_mse(original_image, reconstructed)

        results[defense.name] = (reconstructed, psnr, mse)
        loss_histories[defense.name] = loss_history

        print(f"  PSNR: {psnr:.2f} dB")
        print(f"  MSE: {mse:.4f}")

    print("\n" + "=" * 70)
    print("SUMMARY OF RESULTS")
    print("=" * 70)

    # Print summary table
    print(f"{'Defense Mechanism':<35} {'PSNR (dB)':<15} {'MSE':<15}")
    print("-" * 70)
    for defense_name, (_, psnr, mse) in results.items():
        print(f"{defense_name:<35} {psnr:>12.2f}    {mse:>12.4f}")

    # Visualize results
    print("\nGenerating visualizations...")
    plot_comparison(original_image, results, save_path='./dlg_comparison.png')
    plot_loss_curves(loss_histories, save_path='./dlg_loss_curves.png')

    return results, loss_histories


# ================== Entry Point ==================
if __name__ == "__main__":
    # Set random seed for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)

    # Run experiment
    results, loss_histories = run_experiment(
        sample_idx=0,  # Change this to test different samples
        num_iterations=300,
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )

    print("\n" + "=" * 70)
    print("Experiment completed!")
    print("Results saved to:")
    print("  - dlg_comparison.png")
    print("  - dlg_loss_curves.png")
    print("=" * 70)

Using device: cpu
Original Label: 5

Testing: No Defense
----------------------------------------------------------------------
  Iteration 0, Loss: 100.6587
  Iteration 50, Loss: 0.0000
  Iteration 100, Loss: 0.0000
  Iteration 150, Loss: 0.0000
  Iteration 200, Loss: 0.0000
  Iteration 250, Loss: 0.0000
  PSNR: 60.87 dB
  MSE: 0.0000

Testing: DP (σ=0.5, C=1.0)
----------------------------------------------------------------------
  Iteration 0, Loss: 300089.0000
  Iteration 50, Loss: 299920.4688
  Iteration 100, Loss: 299923.5938
  Iteration 150, Loss: 299921.9062
  Iteration 200, Loss: 299920.7812
  Iteration 250, Loss: 299922.2500
  PSNR: -1.36 dB
  MSE: 1.3680

Testing: DP (σ=1.0, C=1.0)
----------------------------------------------------------------------
  Iteration 0, Loss: 1200058.5000
  Iteration 50, Loss: 1199659.1250
  Iteration 100, Loss: 1199652.5000
  Iteration 150, Loss: 1199648.2500
  Iteration 200, Loss: 1199649.5000
  Iteration 250, Loss: 1199654.5000
  PSNR: -6.30

FileNotFoundError: [Errno 2] No such file or directory: '/home/claude/dlg_comparison.png'